# NTX 17 — Human-Adjudicated Multilingual Authorization

CPU only. Produces a two-rater blind annotation protocol and scores genuine independently adjudicated holdout cases only.

## Scientific safeguard

MASSIVE/MTOP have **intent labels, not authorization labels**. This notebook creates truly blank forms for two independent human raters and evaluates them only after both provide original labels. AI-authored or self-labeled cases are not independent ground truth.

In [ ]:
from pathlib import Path
import sys, os, csv, json, hashlib, zipfile, subprocess, time
import pandas as pd
from datetime import datetime, timezone
KIT = Path('/content/NiyamTraceX_Paper_Final_Validation_Kit') if Path('/content').exists() else Path.cwd().parent
if not (KIT/'evidence').is_dir():
    here = Path.cwd()
    choices = [here, here.parent, Path('/mnt/data/NiyamTraceX_Paper_Final_Validation_Kit')]
    for candidate in choices:
        if (candidate/'evidence').is_dir(): KIT = candidate; break
if not (KIT/'evidence').is_dir():
    package = None
    for p in [Path.cwd()/'NiyamTraceX_Paper_Final_Validation_Kit.zip',Path('/content/NiyamTraceX_Paper_Final_Validation_Kit.zip')]:
        if p.exists(): package=p;break
    if package is None and 'google.colab' in sys.modules:
        from google.colab import files
        print('Upload NiyamTraceX_Paper_Final_Validation_Kit.zip (the full kit).')
        upload = files.upload()
        if 'NiyamTraceX_Paper_Final_Validation_Kit.zip' in upload:
            package = Path.cwd()/'NiyamTraceX_Paper_Final_Validation_Kit.zip'
    if package:
        with zipfile.ZipFile(package) as z:
            for part in z.infolist():
                target=(Path('/content') if Path('/content').exists() else Path.cwd())/part.filename
                if not str(target.resolve()).startswith(str((Path('/content') if Path('/content').exists() else Path.cwd()).resolve())):
                    raise RuntimeError('Unsafe archive path')
            z.extractall('/content' if Path('/content').exists() else '.')
        KIT = (Path('/content') if Path('/content').exists() else Path.cwd())/'NiyamTraceX_Paper_Final_Validation_Kit'
OUT = Path(os.getenv('NTX_PAPER_OUT',str(KIT/'outputs')))
OUT.mkdir(parents=True,exist_ok=True)
print('Kit:',KIT,'\nOutput:',OUT)
assert (KIT/'evidence').is_dir(), 'Evidence not found. Upload the full kit ZIP.'
def dump(obj,p):
    Path(p).parent.mkdir(parents=True,exist_ok=True)
    Path(p).write_text(json.dumps(obj,indent=2,default=str),encoding='utf-8')
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

In [ ]:
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupShuffleSplit
import numpy as np
DATA_FILE=Path(os.environ.get('NTX_NATURAL_CASES',str(KIT/'evidence'/'natural_authorization_cases.csv')))
A1=Path(os.environ.get('NTX_ANNOTATOR_A',str(OUT/'17_annotator_A_completed.csv')))
A2=Path(os.environ.get('NTX_ANNOTATOR_B',str(OUT/'17_annotator_B_completed.csv')))
ADJ=Path(os.environ.get('NTX_ADJUDICATED',str(OUT/'17_adjudicated_completed.csv')))
PRED=Path(os.environ.get('NTX_HOLDOUT_PREDICTIONS',str(OUT/'17_frozen_system_predictions.csv')))
case_cols=['case_id','group_id','language','raw_request','principal_role','tenant_context','policy_context','initial_state_description','source_provenance','consent_status']
label_cols=['case_id','annotator_id','verdict','authorized_tool','authorized_effects_json','ambiguity_type','rationale','annotated_utc']
allowed={'ALLOW','BLOCK','CLARIFY'}
if not DATA_FILE.exists():
    pd.DataFrame(columns=case_cols).to_csv(OUT/'17_natural_cases_TEMPLATE.csv',index=False)
    print('No independent natural cases yet. Blank source template created. No artificial cases/labels generated.')
    cases=pd.DataFrame(columns=case_cols)
else:
    cases=pd.read_csv(DATA_FILE,keep_default_na=False,dtype=str)
    assert set(case_cols).issubset(cases.columns),set(case_cols)-set(cases.columns)
    assert cases.case_id.is_unique and cases.case_id.notna().all() and (cases.case_id!='').all()
    assert all(cases[c].astype(str).str.len().gt(0).all() for c in ['group_id','language','raw_request','policy_context','source_provenance'])
    assert (cases['consent_status']=='approved').all(), 'Need approved and privacy-reviewed cases'
    # Never include gold labels in the source cases or share either rater's labels.
    assert 'verdict' not in cases.columns
    if not A1.exists():pd.DataFrame({'case_id':cases.case_id,'annotator_id':'A'}).reindex(columns=label_cols).to_csv(OUT/'17_annotator_A_BLANK.csv',index=False)
    if not A2.exists():pd.DataFrame({'case_id':cases.case_id,'annotator_id':'B'}).reindex(columns=label_cols).to_csv(OUT/'17_annotator_B_BLANK.csv',index=False)
    (OUT/'17_FROZEN_CASE_SHA256.txt').write_text(sha(DATA_FILE)+'  '+DATA_FILE.name)
    print('Frozen independent cases',len(cases),'languages',sorted(cases.language.unique()))

In [ ]:
# Raters must annotate independently. Expected: one row per case, no missing labels, no shared file.
labels_ready=bool(len(cases)) and A1.is_file() and A2.is_file()
if labels_ready:
    a=pd.read_csv(A1,dtype=str,keep_default_na=False)
    b=pd.read_csv(A2,dtype=str,keep_default_na=False)
    for name,r in [('A',a),('B',b)]:
        assert set(label_cols).issubset(r.columns),(name,set(label_cols)-set(r.columns))
        assert r.case_id.is_unique and set(r.case_id)==set(cases.case_id)
        assert r.verdict.isin(allowed).all(),(name,'invalid/missing verdict')
        assert r.annotator_id.eq(name).all()
        assert r.rationale.astype(str).str.len().gt(0).all()
    pair=a.set_index('case_id').join(b.set_index('case_id'),lsuffix='_A',rsuffix='_B',validate='one_to_one').loc[cases.case_id]
    kappa=float(cohen_kappa_score(pair.verdict_A,pair.verdict_B,labels=sorted(allowed)))
    agreement=float((pair.verdict_A==pair.verdict_B).mean())
    deltas=pair[pair.verdict_A!=pair.verdict_B].reset_index()
    deltas.to_csv(OUT/'17_disagreements_FOR_ADJUDICATION.csv',index=False)
    stats={'n':len(pair),'languages':cases.language.value_counts().to_dict(),'agreement':agreement,'cohen_kappa':kappa,
        'n_disagreements':len(deltas),'A_hash':sha(A1),'B_hash':sha(A2),'independent_rater_confirmation_needed':True}
    dump(stats,OUT/'17_rater_agreement.json')
    print(stats)
else:print('PENDING: independent annotator A and B completed CSVs are required. Zero scores reported.')

In [ ]:
# Gold split and scoring require independent adjudication + FROZEN predictions made without gold labels.
state={'annotation_complete':labels_ready,'adjudication_complete':False,'heldout_scored':False,'blockers':[]}
if not labels_ready:state['blockers'].append('Two independent completed annotation files unavailable')
if not ADJ.exists():state['blockers'].append('Missing adjudicated gold CSV')
if not PRED.exists():state['blockers'].append('Missing frozen holdout prediction CSV')
if labels_ready and ADJ.exists() and PRED.exists():
    gold=pd.read_csv(ADJ,dtype=str,keep_default_na=False)
    pred=pd.read_csv(PRED,dtype=str,keep_default_na=False)
    assert {'case_id','verdict'}.issubset(gold.columns)
    assert {'case_id','decision','variant','model_id','commit_sha','gold_unseen_until_prediction'}.issubset(pred.columns)
    assert gold.case_id.is_unique and set(gold.case_id)==set(cases.case_id)
    assert gold.verdict.isin(allowed).all()
    assert pred.case_id.isin(cases.case_id).all() and pred.decision.isin(allowed).all()
    assert pred.gold_unseen_until_prediction.eq('True').all(), 'Unblinded predictions prohibited'
    assert not pred.duplicated(['case_id','variant','model_id']).any()
    assert (pred.commit_sha.str.len()==40).all()
    joined=pred.merge(gold[['case_id','verdict']],on='case_id',how='left',validate='many_to_one')
    joined=joined.merge(cases[['case_id','group_id','language']],on='case_id',validate='many_to_one')
    # split MUST be sealed before collecting labels/inference; never optimize on heldout outcomes.
    SPLIT=OUT/'17_FROZEN_SPLIT.csv'
    if not SPLIT.exists():
        state['blockers'].append('No sealed pre-annotation split at 17_FROZEN_SPLIT.csv; do not create one post hoc and call it untouched')
    else:
        split=pd.read_csv(SPLIT,dtype=str)
        assert set(split.columns)>={'case_id','split'}
        assert not split.groupby('group_id').split.nunique().gt(1).any() if 'group_id' in split.columns else True
        joined=joined.merge(split[['case_id','split']],on='case_id',validate='many_to_one')
        hold=joined[joined.split=='test'].copy()
        assert len(hold)>0
        summary=hold.assign(correct=hold.decision==hold.verdict).groupby(['model_id','variant','language']).agg(n=('correct','size'),accuracy=('correct','mean')).reset_index()
        summary.to_csv(OUT/'17_heldout_by_language.csv',index=False)
        state['heldout_scored']=True
    state['adjudication_complete']=True
# If cases are available but no split, create a prospective, BLIND design suggestion only (do not call it pre-registered automatically).
if len(cases)>0 and not (OUT/'17_FROZEN_SPLIT.csv').exists():
    split=GroupShuffleSplit(n_splits=1,test_size=.3,random_state=20261011)
    idx=np.arange(len(cases));train,test=next(split.split(idx,groups=cases.group_id))
    proposal=pd.DataFrame({'case_id':cases.case_id,'group_id':cases.group_id,'split':'train'})
    proposal.loc[test,'split']='test'
    proposal.to_csv(OUT/'17_PROPOSED_SPLIT_REVIEW_THEN_FREEZE.csv',index=False)
    print('A proposed split was exported; freeze it BEFORE annotation and predictions in a new run.')
dump(state,OUT/'17_READINESS.json')
print(state)